# FL Phase B — Non-IID Failure-Mode Validation (Kaggle, GPU)

Original (accuracy-only) trust scoring × Dirichlet alpha sweep on MNIST.
Expected: honest clients' trust scores drop as alpha decreases (0.1 → 100),
with NO adversaries — the failure mode motivating the whole paper.

**চালানোর আগে:**
1. Settings → Accelerator → **GPU T4**।
2. Add-ons → Secrets → `KAGGLE_API_TOKEN` যোগ + attach।
3. Cell উপর থেকে নিচে একটা একটা করে Run করুন। Session বন্ধের আগে Cell 8-এর file গুলো download করুন (working dir মুছে যায়)।

In [ ]:
# Cell 1 — environment check
import sys, importlib.util
print('python:', sys.version.split()[0])
for pkg in ['tensorflow', 'sklearn', 'pandas', 'numpy']:
    print(pkg, '=>', importlib.util.find_spec(pkg) is not None)
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 2 — clone (fresh session) বা pull (চলতি session)
import os
if not os.path.isdir('/kaggle/working/Federated-Learning-using-Blockchain'):
    !git clone -b het-aware-trust https://github.com/saimoon-oman/Federated-Learning-using-Blockchain.git
%cd /kaggle/working/Federated-Learning-using-Blockchain
!git pull 2>&1 | tail -n 2
!git log --oneline -1

In [ ]:
# Cell 3 — installs (safe pkgs; tf-privacy fail expected, Phase B non-DP)
!pip install -q fastapi uvicorn pydantic python-multipart 2>&1 | tail -n 1
!pip install -q "tensorflow-privacy==0.9.0" 2>&1 | tail -n 1
import importlib
for pkg in ['fastapi', 'uvicorn', 'multipart', 'pydantic', 'sklearn', 'pandas']:
    try:
        importlib.import_module(pkg)
        print(pkg, 'OK')
    except Exception as e:
        print(pkg, 'FAILED:', repr(e), '<== থামুন, Saimoon-কে পাঠান')
try:
    importlib.import_module('tensorflow_privacy')
    print('tensorflow_privacy OK')
except Exception:
    print('tensorflow_privacy MISSING (expected) — Phase B এগোবে')

In [ ]:
# Cell 4 — Kaggle auth + credit-card CSV (Phase B শুধু MNIST, তবু vefify রাখা)
import os
from kaggle_secrets import UserSecretsClient
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(UserSecretsClient().get_secret('KAGGLE_API_TOKEN'))
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
print('auth OK')

In [ ]:
# Cell 5 — Phase B sweep: trust_based × alpha {0.1, 0.5, 1.0, 100}, MNIST (~25-35 min)
# alpha=100 মানে original split (control)। browser খোলা রাখুন।
!for A in 0.1 0.5 1.0 100; do echo "===== alpha=$A ====="; python python/run_experiment.py --dataset minst --alpha $A --method trust_based --rounds 3 --seed 42; done
!ls -lh python/results/

In [ ]:
# Cell 6 — summary: final accuracy + mean trust_score per alpha
import glob, json
import numpy as np
rows = []
for fp in sorted(glob.glob('python/results/*.jsonl')):
    recs = [json.loads(l) for l in open(fp)]
    last = recs[-1]
    ts = last.get('trust_scores') or []
    rows.append((float(last.get('alpha')), last.get('accuracy'), float(np.mean(ts)) if ts else None, fp))
rows.sort()
print(f"{'alpha':>6} {'final_acc':>10} {'mean_trust':>10}")
for a, acc, mt, fp in rows:
    print(f"{a:>6} {acc:>10.4f} {mt:>10.0f}  {fp.split('/')[-1]}")
print('\nHoenst-client trust alpha-র সাথে কমলে Phase B PASSED (plot দেখুন)')

In [ ]:
# Cell 7 — headline plot: mean honest trust vs alpha
import glob, json
import numpy as np
import matplotlib.pyplot as plt
xs, ys = [], []
for fp in sorted(glob.glob('python/results/*.jsonl')):
    recs = [json.loads(l) for l in open(fp)]
    last = recs[-1]
    ts = last.get('trust_scores') or []
    xs.append(float(last.get('alpha')))
    ys.append(float(np.mean(ts)) if ts else 0)
order = np.argsort(xs)
xs = [xs[i] for i in order]; ys = [ys[i] for i in order]
plt.figure()
plt.plot([str(x) for x in xs], ys, marker='o')
plt.xlabel('alpha (0.1=severe non-IID ... 100=near-IID)')
plt.ylabel('mean client trust score (final round)')
plt.title('Phase B: honest trust vs non-IID severity (trust_based, MNIST)')
plt.grid(True)
plt.savefig('python/results/phase_b_trust_vs_alpha.png', dpi=150)
plt.show()
print('saved python/results/phase_b_trust_vs_alpha.png')

## Cell 8 — ফলাফল পাঠানো (মানুষের কাজ, code নয়)

Session বন্ধের **আগে** বাম পাশের file panel থেকে download করুন:
1. `python/results/*.jsonl` (৪টা file)
2. `python/results/phase_b_trust_vs_alpha.png`

Saimoon-কে পাঠান। প্রত্যাশা: α কমার সাথে mean trust visibly কম — ওটাই paper-এর motivation-figure। Error হলে error-সহ থামুন।